In [7]:
import requests

In [ ]:
from importlib.metadata import metadata


class rag_pipeline:
    def __init__(self, text):
        self.text = text
        self.chunks = self.chunk_text(self.text)


    def get_connection(self):
        import psycopg2
        return psycopg2.connect(
            dbname="vdb",
            user="o6",
            password="",
            host="localhost",
            port="5432"
        )
    
    def chunk_text(self,text, chunk_size=10, overlap=4):
        words = text
        chunks = []
        start = 0
        while start < len(words):
            end = start + chunk_size
            chunk = " ".join(words[start:end])
            chunks.append(chunk)
            start += chunk_size - overlap
        return chunks
    

    def embed_text(self, text):
        OLLAMA_URL = "http://localhost:11434/api/embed"
        MODEL_NAME = "all-minilm"
        response = requests.post(
            OLLAMA_URL,
            json={
                "model": MODEL_NAME,
                "input": text
            }
        )
        embedding = response.json()['embeddings'][0]
        return embedding
    

    def build_vector_store(self):
        for chunk in self.chunks:
            vector = self.embed_text(chunk)
            self.store_vector(vector, chunk)

    def store_vector(self, vector, metadata):
        conn = self.get_connection()
        cur = conn.cursor()

        cur.execute(
            "INSERT INTO documents (content, embedding) VALUES (%s, %s)",
            (metadata, vector)
        )

        conn.commit()
        cur.close()
        conn.close()
            
    
    def search(self, query, top_k=5):
        query_vector = self.embed_text(query)
        query_vector = "[" + ",".join(map(str, query_vector)) + "]"

        conn = self.get_connection()
        cur = conn.cursor()

        cur.execute(
            """
            SELECT content, embedding <=> %s AS distance
            FROM documents
            ORDER BY embedding <=> %s
            LIMIT %s
            """,
            (query_vector, query_vector, top_k)
        )

        results = cur.fetchall()
        
        cur.close()
        conn.close()

        return results
    def model__(self,text):
    
        from groq import Groq

        client = Groq()
        extra_content=self.search(text)
        context = "\n".join([result[0] for result in extra_content])
        print("Context:", context)

        content=f"""Answer the question based ONLY on the following context.
                    If the context doesn't contain enough information, say "I don't have enough information to answer that."

                    Context:
                    {context}

                    Question: {text}

                    Answer:"""
        
        completion = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "user", "content": content}
            ],
            temperature=0.7,
            max_tokens=1024,
            top_p=1,
            stream=True
        )

        for chunk in completion:
            if chunk.choices[0].delta.content:
                print(chunk.choices[0].delta.content, end="")
    

In [51]:
example_txt=words=open('example.txt','r').read().splitlines()

In [ ]:
example_txt[0]

'Krishna Yerukali'

In [ ]:
example_txt[22]

'Experience in using Chrome Developer tools and Firebug for debugging and'

In [52]:
vd=rag_pipeline(example_txt)

In [53]:
vd.build_vector_store()

In [54]:
text='what all technologies  used?'

vd.model__(text)

Context: Extensively used the repositories like GIT & GIT Desktop. ● Experience with various IDEs such as Visual Studio Code, Web Strom, Sublime. Work Experience: ● ● ● ● ● Working as a Senior Angular Developer in Pyramid IT, Noida. From Apr’23 to till date.
electronics, financial services, shipbuilding, and medical services, and two research and development stations that have allowed the chaebol to enter the industries of "high-polymer chemicals, genetic engineering tools aerospace, and nanotechnology. Role & Responsibilities: ● Gathering Application Requirement on Daily Stand-Up call From Business Owner & Analysis. ● Setting-up the project Environment & Application Structure Using Technical resource. ●
Web Technologies : HTML 4/5, XML, JAVASCRIPT, JQUERY , CSS3, SCSS, Angular JS, Angular-2/4/6/7/8/9/11/13/14/15/16/17/18, Angular/material, GitLab CI/CD, Jenkins, Boostrap4, JSON, NgRx, Chart.js, Node.js, J2EE. Web/Application Servers : APACHE TOMCAT, Web sphere, JBOSS, WebLogic. Databa

# Note

## ollama serve -- comment for serving lama

## o6 --- user for the db 